# UMTAM Pruning Experiment: Curvature-Aware vs Magnitude-Only

This notebook reproduces a Figure 2-style experiment (similar to Mahdavinia et al.) for UMTAM.

**Experiment Design:**
1. Fine-tune BERT on multiple GLUE tasks using UMTAM optimizer
2. For each task, compare two pruning strategies at various densities (1%, 5%, 10%, 20%, 40%, 60%, 90%):
   - **UMTAM Curvature-Aware**: Use saliency scores $S^{(i,j)} = (W-W_0)^2 \cdot \sqrt{R \cdot C}$
   - **Magnitude-Only**: Use only $|W-W_0|$
3. Evaluate each pruned model on its task benchmark
4. Plot performance vs density (4-panel figure)

**Expected Result:** UMTAM curvature-aware pruning should outperform magnitude-only, especially at low densities (1-10%).

## Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers datasets torch accelerate evaluate scikit-learn matplotlib seaborn tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import evaluate
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from typing import Dict, List, Tuple, Optional
import json
from pathlib import Path
import math
import copy
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['font.size'] = 10

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## UMTAM Optimizer Implementation

In [ ]:
class UMTAMOptimizer(optim.Optimizer):
    """
    UMTAM: Unified Momentum-Trajectory Aware Training and Merging Optimizer
    Stable version with saliency tracking for pruning experiments
    """
    
    def __init__(
        self,
        params,
        lr: float = 1e-4,
        rank: int = 32,
        beta1: float = 0.9,
        beta2: float = 0.999,
        gamma: float = 0.9,
        eps: float = 1e-8,
        alpha: float = 0.99,
        grad_clip: float = 1.0,
        svd_frequency: int = 10,
        max_precond: float = 10.0,
        max_update_norm: float = 1.0,
        min_stat: float = 1e-4,
        weight_decay: float = 0.0,
    ):
        defaults = dict(
            lr=lr, rank=rank, beta1=beta1, beta2=beta2, gamma=gamma,
            eps=eps, alpha=alpha, grad_clip=grad_clip,
            svd_frequency=svd_frequency, max_precond=max_precond,
            max_update_norm=max_update_norm, min_stat=min_stat,
            weight_decay=weight_decay,
        )
        super(UMTAMOptimizer, self).__init__(params, defaults)
        
    def _init_group_state(self, group, param):
        """Initialize optimizer state for a parameter."""
        state = self.state[param]
        if len(state) > 0:
            return
            
        shape = param.shape
        device = param.device
        dtype = param.dtype
        
        state['step'] = 0
        state['w0'] = param.data.clone().detach()  # Initial weights
        
        if len(shape) == 2:
            m, n = shape
            rank = min(group['rank'], min(m, n))
            
            state['U'] = torch.randn(m, rank, device=device, dtype=dtype) * 0.01
            state['Sigma'] = torch.eye(rank, device=device, dtype=dtype) * 0.01
            state['V'] = torch.randn(n, rank, device=device, dtype=dtype) * 0.01
            state['E'] = torch.zeros_like(param.data)
            
            min_stat = group.get('min_stat', 1e-4)
            state['R'] = torch.ones(m, device=device, dtype=dtype) * min_stat
            state['C'] = torch.ones(n, device=device, dtype=dtype) * min_stat
            state['S'] = torch.zeros_like(param.data)  # Saliency scores
            state['current_rank'] = rank
            
        elif len(shape) == 1:
            state['m'] = torch.zeros_like(param.data)
            state['v'] = torch.zeros_like(param.data)
    
    def _truncated_svd(self, M: torch.Tensor, rank: int):
        """Compute truncated SVD."""
        try:
            U_full, S_full, Vh_full = torch.linalg.svd(M, full_matrices=False)
            actual_rank = min(rank, len(S_full))
            U = U_full[:, :actual_rank]
            S = torch.diag(S_full[:actual_rank])
            V = Vh_full[:actual_rank, :].T
            return U, S, V
        except RuntimeError:
            raise
    
    @torch.no_grad()
    def step(self, closure: Optional = None):
        """Optimization step."""
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        
        for group in self.param_groups:
            for param in group['params']:
                if param.grad is None:
                    continue
                
                self._init_group_state(group, param)
                state = self.state[param]
                grad = param.grad
                state['step'] += 1
                step = state['step']
                
                # Hyperparameters
                lr = group['lr']
                beta1 = group['beta1']
                beta2 = group['beta2']
                gamma = group['gamma']
                eps = group['eps']
                alpha = group['alpha']
                grad_clip = group['grad_clip']
                svd_freq = group['svd_frequency']
                weight_decay = group['weight_decay']
                max_precond = group['max_precond']
                max_update_norm = group['max_update_norm']
                min_stat = group['min_stat']
                
                if weight_decay > 0:
                    param.data.mul_(1 - lr * weight_decay)
                
                # 1D parameters (bias)
                if 'v' in state:
                    m = state['m']
                    v = state['v']
                    
                    grad_norm = torch.norm(grad)
                    if grad_norm > grad_clip:
                        grad = grad * (grad_clip / grad_norm)
                    
                    m.mul_(beta1).add_(grad, alpha=1 - beta1)
                    v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                    
                    bias_correction1 = 1 - beta1 ** step
                    bias_correction2 = 1 - beta2 ** step
                    
                    step_size = lr / bias_correction1
                    denom = (v.sqrt() / math.sqrt(bias_correction2)).add_(eps)
                    param.data.addcdiv_(m, denom, value=-step_size)
                    continue
                
                # 2D parameters
                grad_2d = grad
                param_2d = param.data
                
                # Gradient clipping
                grad_norm = torch.norm(grad_2d, p='fro')
                if grad_norm > grad_clip:
                    grad_2d = grad_2d * (grad_clip / grad_norm)
                
                # Get state
                U = state['U']
                Sigma = state['Sigma']
                V = state['V']
                E = state['E']
                R = state['R']
                C = state['C']
                S = state['S']
                w0 = state['w0']
                
                # Momentum update with error feedback
                M_reconstructed = U @ Sigma @ V.T
                M_tilde = beta1 * M_reconstructed + (1 - beta1) * grad_2d + gamma * E
                
                # SVD
                if step % svd_freq == 0:
                    try:
                        current_rank = state['current_rank']
                        U_new, Sigma_new, V_new = self._truncated_svd(M_tilde, current_rank)
                        state['U'] = U_new
                        state['Sigma'] = Sigma_new
                        state['V'] = V_new
                        U, Sigma, V = U_new, Sigma_new, V_new
                        
                        M_compressed = U @ Sigma @ V.T
                        E_new = M_tilde - M_compressed
                        state['E'] = E_new
                    except RuntimeError:
                        M_compressed = U @ Sigma @ V.T
                else:
                    M_compressed = U @ Sigma @ V.T
                
                # Update second moments
                R_new = beta2 * R + (1 - beta2) * torch.sum(grad_2d ** 2, dim=1)
                C_new = beta2 * C + (1 - beta2) * torch.sum(grad_2d ** 2, dim=0)
                R_new = torch.clamp(R_new, min=min_stat)
                C_new = torch.clamp(C_new, min=min_stat)
                state['R'] = R_new
                state['C'] = C_new
                
                # Construct preconditioner
                S_hat = torch.outer(R_new, C_new) / (torch.sum(R_new) + eps)
                
                # Adaptive regularization
                param_norm = torch.norm(param_2d, p='fro')
                eps_adaptive = eps * max(1.0, grad_norm / param_norm) if param_norm > eps else eps
                
                # Compute preconditioner
                P = 1.0 / torch.sqrt(S_hat + eps_adaptive)
                P = torch.clamp(P, max=max_precond)
                
                # Parameter update
                update = lr * P * M_compressed
                update_norm = torch.norm(update, p='fro')
                if update_norm > max_update_norm:
                    update = update * (max_update_norm / update_norm)
                param_2d.sub_(update)
                
                # Update saliency (KEY: This tracks curvature-weighted importance)
                param_deviation = (param_2d - w0) ** 2
                curvature_weight = torch.sqrt(torch.outer(R_new, C_new))
                S_new = alpha * S + (1 - alpha) * param_deviation * curvature_weight
                state['S'] = S_new
        
        return loss
    
    def get_saliency_scores(self):
        """Retrieve saliency scores for all 2D parameters."""
        saliency_dict = {}
        for group in self.param_groups:
            for param in group['params']:
                if param in self.state and 'S' in self.state[param]:
                    saliency_dict[id(param)] = self.state[param]['S'].clone().detach()
        return saliency_dict
    
    def get_initial_weights(self):
        """Retrieve initial weights (w0) for all parameters."""
        w0_dict = {}
        for group in self.param_groups:
            for param in group['params']:
                if param in self.state and 'w0' in self.state[param]:
                    w0_dict[id(param)] = self.state[param]['w0'].clone().detach()
        return w0_dict

print("✓ UMTAM Optimizer defined")

## Task Configuration

We'll use 4 GLUE tasks similar to the Mahdavinia paper structure:
1. **SST-2**: Sentiment classification (accuracy)
2. **MRPC**: Paraphrase detection (F1)
3. **CoLA**: Linguistic acceptability (Matthews correlation)
4. **QQP**: Question pair similarity (F1)

In [ ]:
TASK_CONFIGS = {
    'sst2': {
        'dataset': 'glue',
        'subset': 'sst2',
        'num_labels': 2,
        'text_keys': ['sentence'],
        'metric': 'accuracy',
        'display_name': 'Sentiment (SST-2)',
    },
    'mrpc': {
        'dataset': 'glue',
        'subset': 'mrpc',
        'num_labels': 2,
        'text_keys': ['sentence1', 'sentence2'],
        'metric': 'f1',
        'display_name': 'Paraphrase (MRPC)',
    },
    'cola': {
        'dataset': 'glue',
        'subset': 'cola',
        'num_labels': 2,
        'text_keys': ['sentence'],
        'metric': 'matthews_correlation',
        'display_name': 'Acceptability (CoLA)',
    },
    'qqp': {
        'dataset': 'glue',
        'subset': 'qqp',
        'num_labels': 2,
        'text_keys': ['question1', 'question2'],
        'metric': 'f1',
        'display_name': 'Question Pairs (QQP)',
    },
}

# Experiment configuration
CONFIG = {
    'model_name': 'bert-base-uncased',
    'max_length': 128,
    'batch_size': 16,
    'num_epochs': 3,  # Quick training for demonstration
    'lr': 2e-5,
    'rank': 32,
    'warmup_steps': 100,
    'max_train_samples': 1000,  # Limit for faster training
    'max_eval_samples': 500,
    'densities': [1, 5, 10, 20, 40, 60, 90],  # Pruning densities to test
}

print(f"Using model: {CONFIG['model_name']}")
print(f"Tasks: {list(TASK_CONFIGS.keys())}")
print(f"Densities to test: {CONFIG['densities']}%")

## Data Loading and Preprocessing

In [ ]:
def load_and_prepare_dataset(task_name: str, tokenizer, max_length: int, 
                            max_train_samples: int = None, max_eval_samples: int = None):
    """Load and tokenize a GLUE dataset."""
    config = TASK_CONFIGS[task_name]
    
    # Load dataset
    dataset = load_dataset(config['dataset'], config['subset'])
    
    # Tokenization function
    def tokenize_function(examples):
        text_keys = config['text_keys']
        if len(text_keys) == 1:
            texts = examples[text_keys[0]]
            return tokenizer(texts, padding='max_length', truncation=True, max_length=max_length)
        else:
            texts1 = examples[text_keys[0]]
            texts2 = examples[text_keys[1]]
            return tokenizer(texts1, texts2, padding='max_length', truncation=True, max_length=max_length)
    
    # Tokenize
    tokenized = dataset.map(tokenize_function, batched=True, remove_columns=dataset['train'].column_names)
    
    # Limit samples
    if max_train_samples:
        tokenized['train'] = tokenized['train'].select(range(min(max_train_samples, len(tokenized['train']))))
    if max_eval_samples and 'validation' in tokenized:
        tokenized['validation'] = tokenized['validation'].select(range(min(max_eval_samples, len(tokenized['validation']))))
    
    return tokenized

print("✓ Data loading functions defined")

## Training Function

In [ ]:
def train_model(model, optimizer, train_dataset, num_epochs: int, batch_size: int, task_name: str):
    """Train model with UMTAM optimizer."""
    model.train()
    model.to(device)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    total_steps = len(train_loader) * num_epochs
    pbar = tqdm(total=total_steps, desc=f"Training {task_name}")
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            # Move to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            pbar.update(1)
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = epoch_loss / len(train_loader)
        print(f"\nEpoch {epoch+1}/{num_epochs} - Avg Loss: {avg_loss:.4f}")
    
    pbar.close()
    return model

print("✓ Training function defined")

## Evaluation Function

In [ ]:
def evaluate_model(model, eval_dataset, batch_size: int, task_name: str):
    """Evaluate model on validation set."""
    model.eval()
    model.to(device)
    
    eval_loader = DataLoader(eval_dataset, batch_size=batch_size)
    
    metric_name = TASK_CONFIGS[task_name]['metric']
    metric = evaluate.load('glue', TASK_CONFIGS[task_name]['subset'])
    
    with torch.no_grad():
        for batch in tqdm(eval_loader, desc=f"Evaluating {task_name}", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1)
            
            metric.add_batch(predictions=predictions.cpu(), references=labels.cpu())
    
    results = metric.compute()
    return results[metric_name]

print("✓ Evaluation function defined")

## Pruning Functions

In [ ]:
def apply_pruning_mask(model, base_model, saliency_scores: Dict, w0_dict: Dict, 
                      density: float, use_curvature: bool = True):
    """
    Apply pruning mask based on either:
    - UMTAM curvature-aware saliency: S = (W-W0)^2 * sqrt(R*C)
    - Magnitude-only: |W-W0|
    
    Returns a new model with pruned task vector.
    """
    pruned_model = copy.deepcopy(base_model)
    
    # Collect all scores and task vectors
    all_scores = []
    param_info = []
    
    for name, param in model.named_parameters():
        param_id = id(param)
        
        # Only process 2D parameters that have saliency scores
        if param_id in saliency_scores and len(param.shape) == 2:
            # Get base weight
            base_param = dict(base_model.named_parameters())[name]
            w0 = w0_dict.get(param_id, base_param.data)
            
            # Compute task vector
            task_vector = param.data - w0
            
            if use_curvature:
                # UMTAM: Use curvature-aware saliency
                scores = saliency_scores[param_id]
            else:
                # Magnitude-only baseline
                scores = torch.abs(task_vector)
            
            # Flatten and store
            scores_flat = scores.flatten().cpu()
            all_scores.append(scores_flat)
            param_info.append({
                'name': name,
                'param': param,
                'base_param': base_param,
                'task_vector': task_vector,
                'w0': w0,
                'shape': param.shape,
                'start_idx': len(torch.cat(all_scores[:-1])) if len(all_scores) > 1 else 0,
            })
    
    # Concatenate all scores
    all_scores_flat = torch.cat(all_scores)
    
    # Compute global threshold
    k = int(len(all_scores_flat) * (density / 100.0))
    threshold = torch.topk(all_scores_flat, k).values[-1] if k > 0 else float('inf')
    
    # Apply masks
    for info in param_info:
        name = info['name']
        task_vector = info['task_vector']
        w0 = info['w0']
        
        # Get scores for this parameter
        if use_curvature:
            param_scores = saliency_scores[id(info['param'])]
        else:
            param_scores = torch.abs(task_vector)
        
        # Create mask
        mask = (param_scores >= threshold).float()
        
        # Apply: W_pruned = W0 + mask * (W - W0)
        pruned_param = dict(pruned_model.named_parameters())[name]
        pruned_param.data = w0 + mask * task_vector
    
    return pruned_model

print("✓ Pruning functions defined")

## Main Experiment Loop

In [ ]:
def run_pruning_experiment(task_name: str, config: Dict):
    """
    Run complete experiment for one task:
    1. Train model with UMTAM
    2. Test pruning at different densities
    3. Compare curvature-aware vs magnitude-only
    """
    print(f"\n{'='*60}")
    print(f"Task: {TASK_CONFIGS[task_name]['display_name']}")
    print(f"{'='*60}\n")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'])
    
    # Load and prepare dataset
    print("Loading dataset...")
    dataset = load_and_prepare_dataset(
        task_name, tokenizer, config['max_length'],
        config['max_train_samples'], config['max_eval_samples']
    )
    
    # Prepare datasets
    train_dataset = dataset['train']
    eval_dataset = dataset['validation'] if 'validation' in dataset else dataset['test']
    
    # Add labels column if not present
    if 'label' in train_dataset.features:
        train_dataset = train_dataset.rename_column('label', 'labels')
        eval_dataset = eval_dataset.rename_column('label', 'labels')
    
    train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    eval_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    
    print(f"Train samples: {len(train_dataset)}")
    print(f"Eval samples: {len(eval_dataset)}")
    
    # Initialize base model (for reference)
    base_model = AutoModelForSequenceClassification.from_pretrained(
        config['model_name'],
        num_labels=TASK_CONFIGS[task_name]['num_labels']
    )
    
    # Initialize model for training
    model = AutoModelForSequenceClassification.from_pretrained(
        config['model_name'],
        num_labels=TASK_CONFIGS[task_name]['num_labels']
    )
    
    # Initialize UMTAM optimizer
    optimizer = UMTAMOptimizer(
        model.parameters(),
        lr=config['lr'],
        rank=config['rank'],
    )
    
    # Train
    print("\nTraining...")
    model = train_model(model, optimizer, train_dataset, 
                       config['num_epochs'], config['batch_size'], task_name)
    
    # Get saliency scores and initial weights
    print("\nExtracting saliency scores...")
    saliency_scores = optimizer.get_saliency_scores()
    w0_dict = optimizer.get_initial_weights()
    print(f"Collected saliency for {len(saliency_scores)} parameters")
    
    # Evaluate full model
    print("\nEvaluating full model...")
    full_performance = evaluate_model(model, eval_dataset, config['batch_size'], task_name)
    print(f"Full model performance: {full_performance:.4f}")
    
    # Test pruning at different densities
    results = {
        'umtam': [],
        'magnitude': [],
        'densities': config['densities'],
        'full_performance': full_performance,
    }
    
    print("\nTesting pruning strategies...")
    for density in config['densities']:
        print(f"\n--- Density: {density}% ---")
        
        # UMTAM curvature-aware
        print("Testing UMTAM curvature-aware...")
        pruned_model_umtam = apply_pruning_mask(
            model, base_model, saliency_scores, w0_dict, density, use_curvature=True
        )
        perf_umtam = evaluate_model(pruned_model_umtam, eval_dataset, config['batch_size'], task_name)
        results['umtam'].append(perf_umtam)
        print(f"  UMTAM: {perf_umtam:.4f}")
        
        # Magnitude-only
        print("Testing magnitude-only...")
        pruned_model_mag = apply_pruning_mask(
            model, base_model, saliency_scores, w0_dict, density, use_curvature=False
        )
        perf_mag = evaluate_model(pruned_model_mag, eval_dataset, config['batch_size'], task_name)
        results['magnitude'].append(perf_mag)
        print(f"  Magnitude: {perf_mag:.4f}")
        
        # Cleanup
        del pruned_model_umtam, pruned_model_mag
        torch.cuda.empty_cache()
    
    return results

print("✓ Main experiment function defined")

## Run Experiments on All Tasks

In [ ]:
# Run experiments
all_results = {}

for task_name in ['sst2', 'mrpc', 'cola', 'qqp']:
    try:
        results = run_pruning_experiment(task_name, CONFIG)
        all_results[task_name] = results
        
        # Save intermediate results
        with open(f'results_{task_name}.json', 'w') as f:
            json.dump(results, f, indent=2)
        
        print(f"\n✓ Completed {task_name}\n")
        
    except Exception as e:
        print(f"\n✗ Error in {task_name}: {e}\n")
        continue

# Save all results
with open('all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("\n✓ All experiments complete!")

## Generate Figure 2-Style Plot

In [ ]:
# Create 2x2 subplot figure
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

task_names = ['sst2', 'mrpc', 'cola', 'qqp']
colors = {'umtam': '#FF6B35', 'magnitude': '#004E89'}

for idx, task_name in enumerate(task_names):
    if task_name not in all_results:
        continue
    
    ax = axes[idx]
    results = all_results[task_name]
    densities = results['densities']
    
    # Plot
    ax.plot(densities, results['umtam'], 'o-', color=colors['umtam'], 
            linewidth=2.5, markersize=8, label='UMTAM (curvature-aware)', alpha=0.9)
    ax.plot(densities, results['magnitude'], 's-', color=colors['magnitude'], 
            linewidth=2.5, markersize=8, label='Magnitude only', alpha=0.9)
    
    # Formatting
    ax.set_xlabel('Density (%)', fontsize=11, fontweight='bold')
    ax.set_ylabel(TASK_CONFIGS[task_name]['metric'].replace('_', ' ').title(), 
                  fontsize=11, fontweight='bold')
    ax.set_title(TASK_CONFIGS[task_name]['display_name'], fontsize=12, fontweight='bold', pad=10)
    ax.legend(loc='lower right', fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(0, 95)
    
    # Add improvement annotations at low densities
    for d_idx in [0, 1]:  # 1% and 5%
        if d_idx < len(densities):
            umtam_val = results['umtam'][d_idx]
            mag_val = results['magnitude'][d_idx]
            if mag_val > 0:
                improvement = ((umtam_val - mag_val) / mag_val) * 100
                if improvement > 5:  # Only show if >5% improvement
                    ax.annotate(f'+{improvement:.1f}%', 
                              xy=(densities[d_idx], umtam_val),
                              xytext=(10, 10), textcoords='offset points',
                              fontsize=8, color=colors['umtam'],
                              bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                                      edgecolor=colors['umtam'], alpha=0.8))

plt.tight_layout()
plt.savefig('umtam_pruning_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('umtam_pruning_comparison.pdf', bbox_inches='tight')
print("\n✓ Figure saved as umtam_pruning_comparison.png and .pdf")
plt.show()

## Summary Statistics

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT SUMMARY")
print("="*60)

for task_name in task_names:
    if task_name not in all_results:
        continue
    
    results = all_results[task_name]
    print(f"\n{TASK_CONFIGS[task_name]['display_name']}:")
    print("-" * 40)
    
    # Full model
    print(f"Full model: {results['full_performance']:.4f}")
    
    # Low density improvements (1%, 5%, 10%)
    print("\nLow-density improvements (UMTAM vs Magnitude):")
    for i, density in enumerate([1, 5, 10]):
        if i < len(results['densities']) and results['densities'][i] == density:
            umtam_val = results['umtam'][i]
            mag_val = results['magnitude'][i]
            if mag_val > 0:
                improvement = ((umtam_val - mag_val) / mag_val) * 100
                print(f"  {density}%: {umtam_val:.4f} vs {mag_val:.4f} ({improvement:+.1f}% improvement)")

print("\n" + "="*60)
print("Experiment complete! Check the plot above for visual results.")
print("="*60)

## Download Results

Run this cell to download all results and the figure.

In [ ]:
from google.colab import files

# Download results
files.download('all_results.json')
files.download('umtam_pruning_comparison.png')
files.download('umtam_pruning_comparison.pdf')

print("✓ Files downloaded!")